# Asistente BDCC
Avance de proyecto 1 · Edgar Avila

Chatbot basado en reglas con Python y NLTK. Diez conversaciones automatizadas, sin participantes humanos. Las salidas conservadas proceden de ejecución local; el notebook es compatible con Google Colab.

Para activar el modo pruebas, hay que cambiar a "False" la variable MODO_INTERACTVO de la ultima celda.

In [ ]:
import sys
import subprocess
assert sys.version_info >= (3, 10)
try:
    import nltk
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "nltk==3.9.2"])
    import nltk
print("Python:", sys.version.split()[0], "| NLTK:", nltk.__version__)

Python: 3.13.16 | NLTK: 3.9.1


In [ ]:
%%writefile chatbot.py
import re
import unicodedata
from nltk.tokenize import RegexpTokenizer

TOKENIZER = RegexpTokenizer(r"\w+")
JUEGOS = {
    "Smash Ultimate": ("smash", "ultimate"),
    "Street Fighter 6": ("street fighter", "sf6"),
    "Fatal Fury": ("fatal fury", "fatal", "cotw"),
}
BASE = {
    "saludo": "Hola, soy el asistente Black Desert Combat Club (BDCC). Consulta juegos, actividades o registro.",
    "club": "BDCC es una comunidad de juegos de pelea en Chihuahua.",
    "juegos": "La base incluye Smash Ultimate, Street Fighter 6 y Fatal Fury.",
    "actividades": "El club organiza torneos presenciales y retas mensuales.",
    "lugar": "No tengo lugar ni horario de un evento cargados. Consulta al staff.",
    "contacto": "Contacta al staff por las redes de Black Desert Combat Club.",
    "despedida": "Gracias por consultar Black Desert. Hasta luego.",
    "cancelar": "Consulta reiniciada. Puedes preguntar por juegos o actividades.",
    "fallback": "Puedo orientar sobre juegos, actividades, registro y contacto.",
}
REGLAS = {
    "cancelar": r"cancelar|cancela|reiniciar",
    "contacto": r"contacto|staff|organizador|redes|humano",
    "inscripcion": r"inscribirme|inscripcion|registrarme|registro|participar",
    "costo": r"precio|costo|cuesta|pagar",
    "lugar": r"lugar|ubicacion|donde|horario|horarios",
    "actividades": r"actividades|retas|cada cuanto|frecuencia",
    "juegos": r"juegos|videojuegos|disciplinas",
    "club": r"que es bdcc|quienes son|que es el club|que es black desert",
    "despedida": r"adios|hasta luego|gracias|salir",
    "saludo": r"hola|buenas|buenos dias",
}


def normalizar(texto):
    texto = unicodedata.normalize("NFD", texto.lower())
    texto = "".join(c for c in texto if not unicodedata.combining(c))
    return " ".join(TOKENIZER.tokenize(texto))


def coincide(patron, texto):
    return bool(re.search(r"\b(?:" + patron + r")\b", texto))


class Chatbot:
    def __init__(self):
        self.juego = None
        self.pendiente = None
        self.historial = []

    def responder(self, mensaje):
        texto = normalizar(mensaje)
        juegos = [nombre for nombre, alias in JUEGOS.items()
                  if any(coincide(re.escape(a), texto) for a in alias)]
        intencion = next((i for i, regla in REGLAS.items()
                          if coincide(regla, texto)), "fallback")
        if intencion == "cancelar":
            self.juego = self.pendiente = None
        elif intencion == "despedida":
            self.pendiente = None
        elif len(juegos) > 1:
            self.juego = None
            if intencion in ("inscripcion", "costo"):
                self.pendiente = intencion
            intencion = "aclaracion"
        else:
            if juegos:
                self.juego = juegos[0]
            if intencion == "fallback" and juegos:
                intencion = self.pendiente or "juegos"
            if intencion in ("inscripcion", "costo"):
                self.pendiente = intencion if not self.juego else None
            elif intencion != "fallback":
                self.pendiente = None

        if intencion == "aclaracion":
            respuesta = "Mencionaste varios juegos. Indica uno para continuar."
        elif intencion in ("inscripcion", "costo") and not self.juego:
            respuesta = "¿Qué juego te interesa: Smash, SF6 o Fatal Fury?"
        elif intencion == "inscripcion":
            respuesta = (f"Para {self.juego}, revisa la convocatoria con el staff. "
                         "Este asistente orienta; no registra inscripciones.")
        elif intencion == "costo":
            respuesta = (f"No tengo una tarifa confirmada para {self.juego}. "
                         "Consulta la convocatoria con el staff.")
        elif intencion == "juegos" and juegos:
            respuesta = f"{self.juego} está incluido en la base del club."
        else:
            respuesta = BASE[intencion]
        resultado = {"mensaje": mensaje, "intencion": intencion,
                     "juego": self.juego, "respuesta": respuesta}
        self.historial.append(resultado)
        return resultado


def conversar():
    bot = Chatbot()
    print(BASE["saludo"])
    while True:
        try:
            resultado = bot.responder(input("Tú: "))
        except (EOFError, KeyboardInterrupt):
            break
        print("BDCC:", resultado["respuesta"])
        if resultado["intencion"] == "despedida":
            break


if __name__ == "__main__":
    conversar()


Overwriting chatbot.py


In [ ]:
%%writefile pruebas.py
import csv
import json
import platform
from pathlib import Path
from time import perf_counter
import nltk
from chatbot import Chatbot

CASOS = [
    ("Presentación", [("Hola", "saludo"), ("¿Qué es BDCC?", "club"),
                       ("Gracias", "despedida")]),
    ("Catálogo", [("¿Qué juegos manejan?", "juegos"), ("SF6", "juegos")]),
    ("Actividades", [("¿Qué actividades realizan?", "actividades"),
                     ("¿Cada cuánto?", "actividades")]),
    ("Registro con contexto", [("Quiero inscribirme", "inscripcion"),
                              ("Smash", "inscripcion"),
                              ("¿Y el costo?", "costo")]),
    ("Costo sin entidad", [("¿Cuál es el precio?", "costo"),
                          ("Fatal Fury", "costo")]),
    ("Datos no disponibles", [("¿Dónde es el evento?", "lugar"),
                             ("¿Cómo contacto al staff?", "contacto")]),
    ("Saludo y solicitud", [("Hola, quiero inscribirme a SF6", "inscripcion"),
                           ("Adiós", "despedida")]),
    ("Cancelación", [("Quiero registrarme", "inscripcion"),
                    ("Cancelar", "cancelar"), ("Juegos", "juegos")]),
    ("Fuera de alcance", [("¿Lloverá mañana?", "fallback"),
                         ("¿Qué horarios tienen?", "lugar")]),
    ("Reformulación", [("¿Qué títulos competitivos ofrecen?", "juegos"),
                      ("¿Qué videojuegos tienen?", "juegos")]),
]


def evaluar():
    filas, conversaciones, tiempos = [], [], []
    for numero, (nombre, turnos) in enumerate(CASOS, 1):
        bot, correctos = Chatbot(), []
        print(f"\nConversación {numero:02d} | {nombre}")
        for mensaje, esperada in turnos:
            inicio = perf_counter()
            salida = bot.responder(mensaje)
            tiempos.append((perf_counter() - inicio) * 1000)
            correcta = salida["intencion"] == esperada
            correctos.append(correcta)
            filas.append({"conversacion": numero, **salida,
                          "esperada": esperada, "correcta": correcta})
            print(f"U: {mensaje}\nB: {salida['respuesta']}")
            print(f"   [{salida['intencion']} | esperado: {esperada}]")
        conversaciones.append(all(correctos))
    resumen = {
        "python": platform.python_version(), "nltk": nltk.__version__,
        "conversaciones": len(CASOS), "mensajes": len(filas),
        "aciertos": sum(f['correcta'] for f in filas),
        "conversaciones_sin_error": sum(conversaciones),
        "fallbacks": sum(f['intencion'] == 'fallback' for f in filas),
        "tiempo_promedio_ms": round(sum(tiempos) / len(tiempos), 3),
    }
    Path("resultados.json").write_text(
        json.dumps({"resumen": resumen, "turnos": filas},
                   ensure_ascii=False, indent=2), encoding="utf-8")
    with open("resultados.csv", "w", encoding="utf-8-sig", newline="") as archivo:
        writer = csv.DictWriter(archivo, fieldnames=filas[0].keys())
        writer.writeheader()
        writer.writerows(filas)
    print("\nRESUMEN", json.dumps(resumen, ensure_ascii=False))
    return resumen


if __name__ == "__main__":
    evaluar()


Overwriting pruebas.py


In [ ]:
%run pruebas.py


Conversación 01 | Presentación
U: Hola
B: Hola, soy el asistente BDCC. Consulta juegos, actividades o registro.
   [saludo | esperado: saludo]
U: ¿Qué es BDCC?
B: BDCC es una comunidad de juegos de pelea en Chihuahua.
   [club | esperado: club]
U: Gracias
B: Gracias por consultar BDCC. Hasta luego.
   [despedida | esperado: despedida]

Conversación 02 | Catálogo
U: ¿Qué juegos manejan?
B: La base incluye Smash Ultimate, Street Fighter 6 y Fatal Fury.
   [juegos | esperado: juegos]
U: SF6
B: Street Fighter 6 está incluido en la base del club.
   [juegos | esperado: juegos]

Conversación 03 | Actividades
U: ¿Qué actividades realizan?
B: El club organiza torneos presenciales y retas mensuales.
   [actividades | esperado: actividades]
U: ¿Cada cuánto?
B: El club organiza torneos presenciales y retas mensuales.
   [actividades | esperado: actividades]

Conversación 04 | Registro con contexto
U: Quiero inscribirme
B: ¿Qué juego te interesa: Smash, SF6 o Fatal Fury?
   [inscripcion | esperad

In [ ]:
from chatbot import Chatbot
bot = Chatbot()
assert bot.responder("")["intencion"] == "fallback"
assert bot.responder("holanda")["intencion"] == "fallback"
assert bot.responder("inscribirme a Smash y SF6")["intencion"] == "aclaracion"
assert bot.responder("SF6")["intencion"] == "inscripcion"
assert bot.responder("costo")["juego"] == "Street Fighter 6"
bot.responder("cancelar")
assert bot.juego is None and bot.pendiente is None
assert Chatbot().juego is None
print("7 verificaciones adicionales correctas: entrada vacía, límites de palabra, ambigüedad, contexto, costo, reinicio y aislamiento.")

7 verificaciones adicionales correctas: entrada vacía, límites de palabra, ambigüedad, contexto, costo, reinicio y aislamiento.


## Conversación interactiva
La variable activa la entrada por consola. La ejecución automática conserva este modo desactivado.

In [ ]:
from chatbot import conversar
MODO_INTERACTIVO = True
if MODO_INTERACTIVO:
    conversar()

Hola, soy el asistente Black Desert Combat Club (BDCC). Consulta juegos, actividades o registro.
